In [1]:
import sys
sys.path.append("../ingestion/python/src")
sys.path.append("../ingestion/python/enrichment_agent")
from silver_enrichment import *
from graph_silver_enrichment import *
from database import Database
from database import Database

import langdetect
import langid
from lingua import Language, LanguageDetectorBuilder
import os

# ── Setup détecteurs ──────────────────────────────────────────────
LANG_MAP = {"en": "en", "es": "es", "fr": "fr", "pt": "pt"}
lingua_languages = [Language.ENGLISH, Language.SPANISH, Language.FRENCH, Language.PORTUGUESE]
lingua_detector = LanguageDetectorBuilder.from_languages(*lingua_languages).build()
LINGUA_MAP = {
    Language.ENGLISH: "en", Language.SPANISH: "es",
    Language.FRENCH: "fr", Language.PORTUGUESE: "pt",
}

def detect_langdetect(text: str) -> str | None:
    try:
        return langdetect.detect(text)
    except Exception:
        return None

def detect_langid(text: str) -> str | None:
    try:
        return langid.classify(text)[0]
    except Exception:
        return None

def detect_lingua(text: str) -> str | None:
    result = lingua_detector.detect_language_of(text)
    return LINGUA_MAP.get(result)

llm = LLM(repr(os.getenv("MISTRAL_TEST")))

# ── Récupère un échantillon d'offres réelles depuis raw.job_offer ─
db = Database()
query = """
    SELECT rjo.id_job, rjo.job_title, rjo.offer_description
    FROM raw.job_offer rjo
    WHERE rjo.offer_description IS NOT NULL
      AND LENGTH(rjo.offer_description) > 50
    ORDER BY rjo.collected_at DESC
    LIMIT 10;
"""
rows = db.execute(query)

# ── Test comparatif titre+description vs description seule ────────
for row in rows:
    title = row["job_title"]
    description = row["offer_description"]
    combined = f"{title} {description}"

    results_combined = {
        "langdetect": detect_langdetect(combined),
        "langid": detect_langid(combined),
        "lingua": detect_lingua(combined),
    }

    llm_result = extract_attributes_node({"job_title": title, "offer_description": description, "is_remote": None})
    print(llm_result)
    llm_langs = llm_result.get("spoken_languages_required") or [None]
    llm_lang = llm_langs[0]
    results_combined["llm"] = llm_lang

    print("=" * 60)
    print(f"titre : {title}")
    ref = results_combined["langdetect"]  # référence pour marquer les divergences
    for name, val in results_combined.items():
        mark = "✅" if val == ref else "❌"
        print(f"  {mark} {name:<12} → {val}")

    divergent = {k: v for k, v in results_combined.items() if v != ref}
    if len(divergent) > 1:  # au moins une vraie divergence en plus de la référence elle-même
        print(f"\n  ⚠️  divergence détectée : {divergent}")

        print("\n  🔍 Vérification sur description seule :")
        desc_results = {
            "langdetect": detect_langdetect(description),
            "langid": detect_langid(description),
            "lingua": detect_lingua(description),
        }
        for name, val in desc_results.items():
            print(f"     ✅ {name:<12} → {val}")

        if llm_lang in desc_results.values():
            print(f"\n  ✅ LLM confirmé par la description ({llm_lang})")
        else:
            print(f"\n  ❌ LLM NON confirmé par la description")

    print("=" * 60)

c:\Users\rolan\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


{'seniority': 'senior', 'is_remote': False, 'contract_type': 'fulltime', 'spoken_languages_required': ['en']}
titre : Senior Data Engineer
  ✅ langdetect   → en
  ✅ langid       → en
  ✅ lingua       → en
  ✅ llm          → en
{'seniority': 'senior', 'is_remote': False, 'contract_type': 'fulltime', 'spoken_languages_required': ['es']}
titre : Sr Analytics Engineer (Buenos Aires)
  ✅ langdetect   → es
  ✅ langid       → es
  ✅ lingua       → es
  ✅ llm          → es
{'seniority': 'mid', 'is_remote': False, 'contract_type': 'fulltime', 'spoken_languages_required': ['en']}
titre : Cloud Data Engineer, Global Services Delivery (Buenos Aires)
  ✅ langdetect   → en
  ✅ langid       → en
  ✅ lingua       → en
  ✅ llm          → en
{'seniority': 'mid', 'is_remote': False, 'contract_type': 'fulltime', 'spoken_languages_required': ['en']}
titre : Forward Deployed Engineer (Buenos Aires)
  ✅ langdetect   → en
  ✅ langid       → en
  ✅ lingua       → en
  ✅ llm          → en
{'seniority': 'junior'

In [2]:
import os
print(repr(os.getenv("MISTRAL_API_KEY")))

'vvzUbQIvBTzeIhz04XH2M55DwK1HOTPK'
